In [1]:
import json
import pandas as pd
import numpy as np
import tqdm as tqdm

import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [2]:
cities = ['regensburg','landshut','bayreuth','schweinfurt','wuerzburg','bamberg']
seed = 42
test_count = 100
seed_idxs = range(1,6)

train_val_configs = ((10, 3),
                     (20, 5),
                     (40, 10),
                     (80, 20),
                     (160, 40))

city_mapping = {
    'regensburg': 'C1',
    'landshut': 'C2',
    'bayreuth': 'C3',
    'schweinfurt': 'C4',
    'wuerzburg': 'C5',
    'bamberg': 'C6'
}

results_dir = '/home/rrao/development/gnn_predicting_effects_of_traffic_policies/data/inductive_gnn_data_results/transductive/Scratch_vs_Finetune/'

metrics = ['top_1_hit_rate', 'top_5_hit_rate', 'top_10_hit_rate', 'bottom_1_hit_rate', 'bottom_5_hit_rate', 'bottom_10_hit_rate',
           'loss', 'r2', 'spearman', 'pearson', 'epochs', 'time']

combined_results = {city: {'scratch': {f"train_{train}_val_{val}": {metric: [] for metric in metrics} for train, val in train_val_configs},
                          'finetune': {f"train_{train}_val_{val}": {metric: [] for metric in metrics} for train, val in train_val_configs}} for city in cities}

random_results = {city: {'scratch': {f"train_{train}_val_{val}": {metric: [] for metric in metrics} for train, val in train_val_configs},
                         'finetune': {f"train_{train}_val_{val}": {metric: [] for metric in metrics} for train, val in train_val_configs}} for city in cities}

In [3]:
# WandB CSV

efficiency_data = pd.read_csv('efficiency.csv')

for row in efficiency_data.itertuples():
    run_name = row.Name

    if "pretrain" in run_name or "INCOMPLETE" in run_name:
        continue
    
    city = run_name.split('_')[0]
    approach = run_name.split('_')[1]
    train_count = int(run_name.split('_')[-2].split('t')[-1])
    val_count = int(run_name.split('_')[-1].split('v')[-1])

    if (train_count, val_count) not in train_val_configs:
        continue

    combined_results[city][approach][f"train_{train_count}_val_{val_count}"]['epochs'].append(row.epoch+1)
    combined_results[city][approach][f"train_{train_count}_val_{val_count}"]['time'].append(row.Runtime/60)  # Convert to minutes

In [4]:
for city in cities:
    for train_count, val_count in train_val_configs:
        for seed_idx in seed_idxs:
            
            scratch_run_name = f"{city}_scratch_rs_{seed_idx}_t{train_count}_v{val_count}"
            finetune_run_name = f"{city}_finetune_rs_{seed_idx}_t{train_count}_v{val_count}"
            results_json_name = f"{city}_rs{seed_idx}_t{train_count}_v{val_count}_seed{seed+seed_idx-1}_train{train_count}_val{val_count}_test{test_count}_distant_iou_metrics.json"
            random_results_json_name = f"{city}_rs{seed_idx}_t{train_count}_v{val_count}_seed{seed+seed_idx-1}_train{train_count}_val{val_count}_test{test_count}_random_metrics.json"

            with open(results_dir + scratch_run_name + '/evaluation/' + results_json_name, 'r') as f:
                scratch_results = json.load(f)
            
            with open(results_dir + finetune_run_name + '/evaluation/' + results_json_name, 'r') as f:
                finetune_results = json.load(f)

            with open(results_dir + scratch_run_name + '/evaluation/' + random_results_json_name, 'r') as f:
                scratch_random_results = json.load(f)

            with open(results_dir + finetune_run_name + '/evaluation/' + random_results_json_name, 'r') as f:
                finetune_random_results = json.load(f)

            for metric in metrics:

                if metric in ['epochs', 'time']:
                    continue  # Already recorded from WandB CSV
                
                combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"][metric].append(scratch_results['hit_rates'][metric] if 'hit_rate' in metric else scratch_results[metric])
                combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"][metric].append(finetune_results['hit_rates'][metric] if 'hit_rate' in metric else finetune_results[metric])

                random_results[city]['scratch'][f"train_{train_count}_val_{val_count}"][metric].append(scratch_random_results['hit_rates'][metric] if 'hit_rate' in metric else scratch_random_results[metric])
                random_results[city]['finetune'][f"train_{train_count}_val_{val_count}"][metric].append(finetune_random_results['hit_rates'][metric] if 'hit_rate' in metric else finetune_random_results[metric])

### TABLES!

In [ ]:
def calc_diff(scratch_x, finetune_x,
              absolute_diff=True, metric_to_perc=False):

      scratch_arr = np.array(scratch_x)*100 if metric_to_perc else np.array(scratch_x)
      finetune_arr = np.array(finetune_x)*100 if metric_to_perc else np.array(finetune_x)

      # diff = finetune_arr - scratch_arr
      # if not absolute_diff:
      #       diff = (diff / scratch_arr) * 100
      
      # return f"{np.mean(diff):.2f} $\\pm$ {np.std(diff):.2f}"

      diff = np.mean(finetune_arr) - np.mean(scratch_arr)
      if not absolute_diff:
            diff = (diff / np.mean(scratch_arr)) * 100

      return f"{diff:.2f}"

In [ ]:
# Results Table
train_count, val_count = train_val_configs[2]

for city in cities:
    
    print(f"\\multirow{{4}}{{*}}{{{city_mapping[city]}}}")
    
    scratch_results = combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"]
    finetune_results = combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"]

    print(f"& Scratch & "
          f"{np.mean(scratch_results['loss']):.2f} $\\pm$ {np.std(scratch_results['loss']):.2f} & "
          f"{np.mean(scratch_results['top_1_hit_rate'])*100:.2f} $\\pm$ {np.std(scratch_results['top_1_hit_rate'])*100:.2f} & "
          f"{np.mean(scratch_results['bottom_1_hit_rate'])*100:.2f} $\\pm$ {np.std(scratch_results['bottom_1_hit_rate'])*100:.2f} & "
          f"{np.mean(scratch_results['r2']):.2f} $\\pm$ {np.std(scratch_results['r2']):.2f} & "
          f"{np.mean(scratch_results['spearman']):.2f} $\\pm$ {np.std(scratch_results['spearman']):.2f} & "
          f"{np.mean(scratch_results['pearson']):.2f} $\\pm$ {np.std(scratch_results['pearson']):.2f} \\\\")
    
    print(f"& Finetune & "
          f"{np.mean(finetune_results['loss']):.2f} $\\pm$ {np.std(finetune_results['loss']):.2f} & "
          f"{np.mean(finetune_results['top_1_hit_rate'])*100:.2f} $\\pm$ {np.std(finetune_results['top_1_hit_rate'])*100:.2f} & "
          f"{np.mean(finetune_results['bottom_1_hit_rate'])*100:.2f} $\\pm$ {np.std(finetune_results['bottom_1_hit_rate'])*100:.2f} & "
          f"{np.mean(finetune_results['r2']):.2f} $\\pm$ {np.std(finetune_results['r2']):.2f} & "
          f"{np.mean(finetune_results['spearman']):.2f} $\\pm$ {np.std(finetune_results['spearman']):.2f} & "
          f"{np.mean(finetune_results['pearson']):.2f} $\\pm$ {np.std(finetune_results['pearson']):.2f} \\\\")
    
    print(f"& Absolute $\\Delta$ & "
          f"{calc_diff(scratch_results['loss'], finetune_results['loss'])} & "
          f"{calc_diff(scratch_results['top_1_hit_rate'], finetune_results['top_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_1_hit_rate'], finetune_results['bottom_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['r2'], finetune_results['r2'])} & "
          f"{calc_diff(scratch_results['spearman'], finetune_results['spearman'])} & "
          f"{calc_diff(scratch_results['pearson'], finetune_results['pearson'])} \\\\")
    
    print(f"& Relative $\\Delta$ & "
          f"{calc_diff(scratch_results['loss'], finetune_results['loss'], absolute_diff=False)} & "
          f"{calc_diff(scratch_results['top_1_hit_rate'], finetune_results['top_1_hit_rate'], absolute_diff=False, metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_1_hit_rate'], finetune_results['bottom_1_hit_rate'], absolute_diff=False, metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['r2'], finetune_results['r2'], absolute_diff=False)} & "
          f"{calc_diff(scratch_results['spearman'], finetune_results['spearman'], absolute_diff=False)} & "
          f"{calc_diff(scratch_results['pearson'], finetune_results['pearson'], absolute_diff=False)} \\\\")
    
    print("\\midrule")

In [ ]:
# Bring back later if needed
print(f"{np.mean(scratch_results['epochs']):.2f} $\\pm$ {np.std(scratch_results['epochs']):.2f} & "
      f"{np.mean(scratch_results['time']):.2f} $\\pm$ {np.std(scratch_results['time']):.2f} & ")

In [ ]:
# Performance Difference: Distant vs Random Test Sets
train_count, val_count = train_val_configs[2]

for city in cities:
    print("\\midrule")
    
    scratch_results = combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"]
    finetune_results = combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"]
    random_scratch_results = random_results[city]['scratch'][f"train_{train_count}_val_{val_count}"]
    random_finetune_results = random_results[city]['finetune'][f"train_{train_count}_val_{val_count}"]

    print(f"{city_mapping[city]} & Scratch & "
          f"{calc_diff(scratch_results['top_1_hit_rate'], random_scratch_results['top_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_1_hit_rate'], random_scratch_results['bottom_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['top_5_hit_rate'], random_scratch_results['top_5_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_5_hit_rate'], random_scratch_results['bottom_5_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['top_10_hit_rate'], random_scratch_results['top_10_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['bottom_10_hit_rate'], random_scratch_results['bottom_10_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(scratch_results['loss'], random_scratch_results['loss'])} & "
          f"{calc_diff(scratch_results['r2'], random_scratch_results['r2'])} & "
          f"{calc_diff(scratch_results['spearman'], random_scratch_results['spearman'])} & "
          f"{calc_diff(scratch_results['pearson'], random_scratch_results['pearson'])} \\\\")
    
    print(f"{city_mapping[city]} & Finetune & "
          f"{calc_diff(finetune_results['top_1_hit_rate'], random_finetune_results['top_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['bottom_1_hit_rate'], random_finetune_results['bottom_1_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['top_5_hit_rate'], random_finetune_results['top_5_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['bottom_5_hit_rate'], random_finetune_results['bottom_5_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['top_10_hit_rate'], random_finetune_results['top_10_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['bottom_10_hit_rate'], random_finetune_results['bottom_10_hit_rate'], metric_to_perc=True)} & "
          f"{calc_diff(finetune_results['loss'], random_finetune_results['loss'])} & "
          f"{calc_diff(finetune_results['r2'], random_finetune_results['r2'])} & "
          f"{calc_diff(finetune_results['spearman'], random_finetune_results['spearman'])} & "
          f"{calc_diff(finetune_results['pearson'], random_finetune_results['pearson'])} \\\\")

In [ ]:
# Hit Rates Table
train_count, val_count = train_val_configs[2]

for direction in ["top", "bottom"]:
    for k in [1, 5, 10]:
        metric = f"{direction}_{k}_hit_rate"
        print("\\midrule")
        for city in cities:
            scratch_values = combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"][metric]
            finetune_values = combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"][metric]
            diff_values = np.array(finetune_values) - np.array(scratch_values)

            print(f"{direction.capitalize()} & {k} & {city_mapping[city]} & "
                  f"{np.mean(scratch_values)*100:.2f} $\\pm$ {np.std(scratch_values)*100:.2f} & "
                  f"{np.mean(finetune_values)*100:.2f} $\\pm$ {np.std(finetune_values)*100:.2f} & "
                  f"{np.mean(diff_values)*100:.2f} $\\pm$ {np.std(diff_values)*100:.2f} \\\\")

### Test Distances

In [ ]:
def compute_test_distances(train_count, val_count, test_set_type="distant_iou"):
    
    test_distances = {city: [] for city in cities}

    for city in cities:
        for seed_idx in seed_idxs:
            
            test_split_json_path = f'/home/rrao/development/gnn_predicting_effects_of_traffic_policies/data/splits/{city}/rs_{seed_idx}/t{train_count}_v{val_count}/'
            test_split_json_path += f'{city}_rs{seed_idx}_t{train_count}_v{val_count}_seed{42+seed_idx-1}_train{train_count}_val{val_count}_test{test_count}_{test_set_type}.json'

            with open(test_split_json_path, 'r') as f:
                test_split = json.load(f)

            # Multiple ways here!
            # test_distances_when_picked = np.array(test_split["test_distances_when_picked"])
            
            test_distances_from_train = np.array(test_split["test_distances_from_train"])
            test_distances_from_val = np.array(test_split["test_distances_from_val"])
            
            # test_distances_combi = np.minimum(test_distances_from_train, test_distances_from_val)
            test_distances_combi = 0.8 * test_distances_from_train + 0.2 * test_distances_from_val

            test_distances[city].extend(test_distances_combi.tolist())

    return test_distances

In [ ]:
iou_test_distances = compute_test_distances(train_count=40, val_count=10, test_set_type="distant_iou")
random_test_distances = compute_test_distances(train_count=40, val_count=10, test_set_type="random")

In [ ]:
print("IOU Test Distances:")
for city in cities:
    distances = iou_test_distances[city]
    print(f"{city.capitalize()}: {np.mean(distances):.6f} ± {np.std(distances):.6f}")

print("\nRandom Test Distances:")
for city in cities:
    distances = random_test_distances[city]
    print(f"{city.capitalize()}: {np.mean(distances):.6f} ± {np.std(distances):.6f}")

### Correlation plots (outdated!)

In [ ]:
color_map = {
    'Scratch: Top 1': 'blue',
    'Finetune: Top 1': 'green',
    'Scratch: Top 5': 'red',
    'Finetune: Top 5': 'orange',}

In [ ]:
def plot_sample_efficiency(metrics, city):

    fig = plt.figure(figsize=(10, 6))

    for metric in metrics:
        
        scratch_points = []
        finetune_points = []

        for train_count, val_count in train_val_configs:
            
            scratch_values = combined_results[city]['scratch'][f"train_{train_count}_val_{val_count}"][metric]
            finetune_values = combined_results[city]['finetune'][f"train_{train_count}_val_{val_count}"][metric]
            
            scratch_points.extend([(train_count + val_count), val] for val in scratch_values)
            finetune_points.extend([(train_count + val_count), val] for val in finetune_values)

        scratch_label = 'Scratch: ' + metric.split('_hit_rate')[0].replace('_', ' ').capitalize()
        finetune_label = 'Finetune: ' + metric.split('_hit_rate')[0].replace('_', ' ').capitalize()

        for points, label, color in [(scratch_points, scratch_label, color_map[scratch_label]),
                                     (finetune_points, finetune_label, color_map[finetune_label])]:
            x, y = zip(*points)
            plt.scatter(x, y, label=label, alpha=0.7, color=color)

        # Fit and plot regression lines
        for points, color in [(scratch_points, color_map[scratch_label]),
                              (finetune_points, color_map[finetune_label])]:
            x, y = zip(*points)
            x = np.array(x).reshape(-1, 1)
            y = np.array(y)
            model = LinearRegression()
            model.fit(x, y)
            x_range = np.linspace(min(x), max(x), 100).reshape(-1, 1)
            y_pred = model.predict(x_range)
            plt.plot(x_range, y_pred, color=color, linestyle='--', alpha=0.7)

    plt.xlabel('Number of Samples')
    # plt.ylabel(metric.replace('_', ' ').capitalize())
    plt.ylabel('Hit Rate')
    plt.title(f'Sample efficiency for {city.capitalize()}')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    # plt.savefig(f'plots/sample_efficiency/{city}_{metric}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'plots/sample_efficiency/{city}.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
for direction in ["top", "bottom"]:
    for k in [1, 5, 10]:
        metric = f"{direction}_{k}_hit_rate"
        for city in cities:
            plot_sample_efficiency(metric, city)

In [ ]:
plot_sample_efficiency(['top_1_hit_rate', 'top_5_hit_rate'], 'schweinfurt')